# TSTR GM Dataset A - Diabetes

In [1]:
#import libraries
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import os
print('Libraries imported!!')

Libraries imported!!


In [2]:
#define directory of functions and actual directory
FUNCTIONS_HOME = '../../../functions/evaluation_functions/' #home directory of the project
REAL_DATA_HOME = '../../../data/raw/chap/' #home directory of the project
SYN_DATA_HOME  = '../../../data/processed/chap/' #home directory of the project
FUNCTIONS_DIR = 'EVALUATION FUNCTIONS/UTILITY'
ACTUAL_DIR = os.getcwd()

#change directory to functions directory
os.chdir(FUNCTIONS_HOME + FUNCTIONS_DIR)

#import functions for data labelling analisys
from utility_evaluation import DataPreProcessor
from utility_evaluation import train_evaluate_model

#change directory to actual directory
os.chdir(ACTUAL_DIR)
print('Functions imported!!')

Functions imported!!


## 1. Read data

In [3]:
#read real dataset
train_data = pd.read_csv(SYN_DATA_HOME + '1_Chap_Data_Synthetic_GM.csv')
categorical_columns = ['group']
for col in categorical_columns :
    train_data[col] = train_data[col].astype('category')
train_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,2.380848,1.437030,-0.272048,0.040346,2.832409,0.493765,0.076030,0.063044
1,Group0,1.298324,-0.872445,-0.199532,0.152709,-2.241116,-0.446154,0.230793,0.003883
2,Group0,-1.418017,-0.705664,0.241301,-0.045601,-4.902270,-1.270940,-0.013372,-0.047719
3,Group1,-4.099877,-0.001948,0.106520,0.082828,-5.664343,-0.253154,-0.025399,0.262347
4,Group0,0.913321,0.179957,-0.291470,0.143974,-3.917341,-0.663543,0.021135,0.134408
...,...,...,...,...,...,...,...,...,...
2712,Group0,-6.513359,-1.122108,0.402461,-0.092153,-3.640159,-0.628105,0.345454,-0.211614
2713,Group0,5.728211,0.579826,0.396367,-0.133686,8.010616,0.676937,0.013272,-0.012647
2714,Group0,-1.149620,-0.016340,-0.443537,-0.220429,-1.130278,-0.292354,-0.017134,-0.058395
2715,Group0,6.866358,1.290890,-0.499429,0.124473,6.232146,0.600912,0.018399,0.237372


In [4]:
#read test data
test_data = pd.read_csv(REAL_DATA_HOME + '1_Chap_Data_Real_Test.csv')
for col in categorical_columns :
    test_data[col] = test_data[col].astype('category')
test_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,-3.053475,-0.177497,-0.010661,0.205118,-2.773102,-0.653440,0.319585,0.017457
1,Group0,-1.375254,0.761393,0.270692,0.140597,-2.157208,-0.297755,0.126143,0.118270
2,Group0,3.681521,1.147414,-0.119372,0.276912,1.452468,0.068735,0.043671,0.074436
3,Group0,0.383070,0.211771,0.374537,0.254718,5.352641,-0.611636,0.344336,0.044627
4,Group0,2.009235,0.765321,0.100557,0.058659,2.842860,0.980271,-0.014527,0.096506
...,...,...,...,...,...,...,...,...,...
674,Group0,1.185955,-0.248240,-0.258972,0.284599,0.816753,0.553623,-0.456024,0.157975
675,Group0,-4.898967,-0.576216,-0.150236,0.069086,-4.450551,-0.192307,0.034382,-0.174072
676,Group0,-3.339095,0.856460,-1.021265,-0.131611,0.639099,0.783467,-0.128578,-0.067689
677,Group0,-3.844234,0.083773,0.334898,-0.210940,-1.259774,0.726944,-0.299066,-0.123455


In [5]:
target = 'group'
#quick look at the breakdown of class values
print('Train data')
print(train_data.shape)
print(train_data.groupby(target).size())
print('#####################################')
print('Test data')
print(test_data.shape)
print(test_data.groupby(target).size())

Train data
(2717, 9)
group
Group0    2627
Group1      90
dtype: int64
#####################################
Test data
(679, 9)
group
Group0    645
Group1     34
dtype: int64


## 2. Pre-process training data

In [6]:
target = 'group'
categorical_columns = []
numerical_columns = train_data.select_dtypes(include=['int64','float64']).columns.tolist()
categories = [np.array(range(2))] if categorical_columns else []
data_preprocessor = DataPreProcessor(categorical_columns, numerical_columns, categories)
x_train = data_preprocessor.preprocess_train_data(train_data.loc[:, train_data.columns != target])
y_train = train_data.loc[:, target]

x_train.shape, y_train.shape

((2717, 8), (2717,))

## 3. Preprocess test data

In [7]:
x_test = data_preprocessor.preprocess_test_data(test_data.loc[:, test_data.columns != target])
y_test = test_data.loc[:, target]
x_test.shape, y_test.shape

((679, 8), (679,))

## 4. Create a dataset to save the results

In [8]:
results = pd.DataFrame(columns = ['model','accuracy','precision','recall','f1'])
results

,model,accuracy,precision,recall,f1


## 4. Train and evaluate Random Forest Classifier

In [9]:
rf_results = train_evaluate_model('RF', x_train, y_train, x_test, y_test)
results = pd.concat([results, rf_results], ignore_index=True)
rf_results

[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:    0.0s
[Parallel(n_jobs=3)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:    0.0s
[Parallel(n_jobs=3)]: Done 100 out of 100 | elapsed:    0.0s finished


,model,accuracy,precision,recall,f1
0,RF,0.9499,0.9024,0.9499,0.9255


## 5. Train and Evaluate KNeighbors Classifier

In [10]:
knn_results = train_evaluate_model('KNN', x_train, y_train, x_test, y_test)
results = pd.concat([results, knn_results], ignore_index=True)
knn_results

,model,accuracy,precision,recall,f1
0,KNN,0.9499,0.9024,0.9499,0.9255


## 6. Train and evaluate Decision Tree Classifier

In [11]:
dt_results = train_evaluate_model('DT', x_train, y_train, x_test, y_test)
results = pd.concat([results, dt_results], ignore_index=True)
dt_results

,model,accuracy,precision,recall,f1
0,DT,0.9234,0.9082,0.9234,0.9156


## 7. Train and evaluate Support Vector Machines Classifier

In [12]:
svm_results = train_evaluate_model('SVM', x_train, y_train, x_test, y_test)
results = pd.concat([results, svm_results], ignore_index=True)
svm_results

[LibSVM]WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -817.257785, rho = 0.941132
nSV = 195, nBSV = 0
Total nSV = 195
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -958.074340, rho = -0.081333
nSV = 192, nBSV = 0
Total nSV = 192
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -925.130141, rho = 1.261599
nSV = 201, nBSV = 0
Total nSV = 201
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -1052.018945, rho = 0.206083
nSV = 196, nBSV = 0
Total nSV = 196
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -882.085721, rho = 1.069648
nSV = 185, nBSV = 0
Total nSV = 185
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -219.378871, rho = -0.249577
nSV = 128, nBSV = 0
Total nSV = 128


,model,accuracy,precision,recall,f1
0,SVM,0.6583,0.9069,0.6583,0.7546


## 8. Train and evaluate Multilayer Perceptron Classifier

In [13]:
mlp_results = train_evaluate_model('MLP', x_train, y_train, x_test, y_test)
results = pd.concat([results, mlp_results], ignore_index=True)
mlp_results

Iteration 1, loss = 0.52999079
Iteration 2, loss = 0.23978973
Iteration 3, loss = 0.15991510
Iteration 4, loss = 0.15082150
Iteration 5, loss = 0.14316914
Iteration 6, loss = 0.14015714
Iteration 7, loss = 0.13786570
Iteration 8, loss = 0.13612953
Iteration 9, loss = 0.13447652
Iteration 10, loss = 0.13306927
Iteration 11, loss = 0.13192669
Iteration 12, loss = 0.13094268
Iteration 13, loss = 0.12973995
Iteration 14, loss = 0.12961515
Iteration 15, loss = 0.12828997
Iteration 16, loss = 0.12748732
Iteration 17, loss = 0.12662991
Iteration 18, loss = 0.12660851
Iteration 19, loss = 0.12578928
Iteration 20, loss = 0.12562505
Iteration 21, loss = 0.12480917
Iteration 22, loss = 0.12380309
Iteration 23, loss = 0.12311902
Iteration 24, loss = 0.12256952
Iteration 25, loss = 0.12140166
Iteration 26, loss = 0.12109181
Iteration 27, loss = 0.12054042
Iteration 28, loss = 0.11996082
Iteration 29, loss = 0.11945465
Iteration 30, loss = 0.11894174
Iteration 31, loss = 0.11898398
Iteration 32, los

,model,accuracy,precision,recall,f1
0,MLP,0.9426,0.9104,0.9426,0.9242


## 9. Save results file

In [14]:
results.to_csv('RESULTS/models_results_gm.csv', index=False)
results

,model,accuracy,precision,recall,f1
0,RF,0.9499,0.9024,0.9499,0.9255
1,KNN,0.9499,0.9024,0.9499,0.9255
2,DT,0.9234,0.9082,0.9234,0.9156
3,SVM,0.6583,0.9069,0.6583,0.7546
4,MLP,0.9426,0.9104,0.9426,0.9242
